In [6]:
from bs4 import BeautifulSoup
import json
import html

arquivo_html = "Lab02Tarefa03.html"
arquivo_saida = "Lab03_Tarefa03_com_outputs.ipynb"

with open(arquivo_html, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f.read(), "html.parser")

celulas = []

main = soup.find("main")
if main is None:
    raise ValueError("Não encontrei a tag <main> no HTML.")

# pega as células na ordem real do notebook
elementos = main.find_all("div", class_="jp-Cell", recursive=False)

for elem in elementos:
    classes = elem.get("class", [])

    # -------------------------
    # CÉLULA MARKDOWN
    # -------------------------
    if "jp-MarkdownCell" in classes:
        md = elem.find("div", class_="jp-RenderedMarkdown")
        if md:
            texto = md.get_text("\n", strip=False)
            texto = html.unescape(texto).strip()

            if texto:
                celulas.append({
                    "cell_type": "markdown",
                    "metadata": {},
                    "source": [linha + "\n" for linha in texto.splitlines()]
                })

    # -------------------------
    # CÉLULA DE CÓDIGO
    # -------------------------
    elif "jp-CodeCell" in classes:
        bloco_codigo = elem.find("div", class_="highlight")
        if not bloco_codigo:
            continue

        codigo = bloco_codigo.get_text("\n", strip=False)
        codigo = html.unescape(codigo).strip()

        if not codigo:
            continue

        outputs = []

        # tenta capturar saídas de texto
        areas_saida = elem.find_all("div", class_="jp-OutputArea-output")

        for area in areas_saida:
            texto_saida = area.get_text("\n", strip=False)
            texto_saida = html.unescape(texto_saida).strip()

            if texto_saida:
                outputs.append({
                    "output_type": "stream",
                    "name": "stdout",
                    "text": [linha + "\n" for linha in texto_saida.splitlines()]
                })

        celulas.append({
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": outputs,
            "source": [linha + "\n" for linha in codigo.splitlines()]
        })

notebook = {
    "cells": celulas,
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python",
            "version": "3.x"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 5
}

with open(arquivo_saida, "w", encoding="utf-8") as f:
    json.dump(notebook, f, ensure_ascii=False, indent=2)

print(f"Notebook salvo como: {arquivo_saida}")
print(f"Total de células reconstruídas: {len(celulas)}")

Notebook salvo como: Lab03_Tarefa03_com_outputs.ipynb
Total de células reconstruídas: 7
